In [2]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.instantiate();

using LinearAlgebra
using ForwardDiff
using PyPlot
using Rotations
using Debugger

  Activating project at `~/workspace/myProject/julia_ws`


In [3]:
# robot data
struct Robot
    Tlist_init::Array
    axis_list::Array
    Tlist::Array
end

In [4]:
# Generate a manipulator
function generate_manipulate()
    Tlist_init = []
    axis_list = []
    T0 = [1 0 0 0.2; 0 1 0 0; 0 0 1 0; 0 0 0 1]
    axis0 = [0; 0; 1]

    T1 = [1 0 0 0; 0 1 0 0; 0 0 1 0.1; 0 0 0 1]
    axis1 = [0; 1; 0]

    T2 = [1 0 0 0; 0 1 0 0.2; 0 0 1 0; 0 0 0 1]
    axis2 = [1; 0; 0]

    T3 = [1 0 0 0; 0 1 0 0; 0 0 1 0.2; 0 0 0 1]
    axis3 = [0; 1; 0]

    T4 = [1 0 0 0; 0 1 0 0; 0 0 1 0.3; 0 0 0 1]
    axis4 = [0; 1; 0]

    T5 = [1 0 0 0; 0 1 0 0; 0 0 1 0.1; 0 0 0 1]
    axis5 = [0; 0; 1]

    push!(Tlist_init, T0)
    push!(Tlist_init, T1)
    push!(Tlist_init, T2)
    push!(Tlist_init, T3)
    push!(Tlist_init, T4)
    push!(Tlist_init, T5)
    push!(axis_list, axis0)
    push!(axis_list, axis1)
    push!(axis_list, axis2)
    push!(axis_list, axis3)
    push!(axis_list, axis4)
    push!(axis_list, axis5)
    return Tlist_init, axis_list
end

generate_manipulate (generic function with 1 method)

In [5]:

function Rotm2T4(rotm)
    T = zeros(4, 4)
    T[1:3, 1:3] = rotm
    T[4, 4] = 1
    return T
end


Rotm2T4 (generic function with 1 method)

In [6]:
## Test
rotm = [1 2 3; 1 1 1; 1 0 0]
T = Rotm2T4(rotm)

4×4 Matrix{Float64}:
 1.0  2.0  3.0  0.0
 1.0  1.0  1.0  0.0
 1.0  0.0  0.0  0.0
 0.0  0.0  0.0  1.0

In [7]:
## forward kinematics 
function fkine(robot, q)
    if length(q) != length(robot.Tlist_init)
        error("The length of q should be equal to the length of robot.Tlist_init")
    end
    println(length(q))
    for i in 1:length(q)
        rot_vec = normalize(robot.axis_list[i]) * q[i]
        rotm = RotationVec(rot_vec[1], rot_vec[2], rot_vec[3])
        if i <= 1
            robot.Tlist[i] = robot.Tlist_init[i] * Rotm2T4(rotm)
        else
            robot.Tlist[i] = robot.Tlist[i-1] * robot.Tlist_init[i] * Rotm2T4(rotm)
        end
        # println("i:", i, "Tlist", robot.Tlist[i])
    end
    return robot
end


fkine (generic function with 1 method)

In [9]:
Tlist, Axis_list = generate_manipulate()
arm = Robot(Tlist, Axis_list, Tlist)
qpos = [pi / 2, pi / 2, 0, 0, 0, 0]
a = 1
arm = fkine(arm, qpos)
arm.Tlist[6][1:3, 4]

6
i:1Tlist[2.220446049250313e-16 -1.0 0.0 0.2; 1.0 2.220446049250313e-16 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]
i:2Tlist[4.930380657631324e-32 -1.0 2.220446049250313e-16 0.2; 2.220446049250313e-16 2.220446049250313e-16 1.0 0.0; -1.0 0.0 2.220446049250313e-16 0.1; 0.0 0.0 0.0 1.0]
i:3Tlist[4.930380657631324e-32 -1.0 2.220446049250313e-16 0.0; 2.220446049250313e-16 2.220446049250313e-16 1.0 4.4408920985006264e-17; -1.0 0.0 2.220446049250313e-16 0.1; 0.0 0.0 0.0 1.0]
i:4Tlist[4.930380657631324e-32 -1.0 2.220446049250313e-16 4.4408920985006264e-17; 2.220446049250313e-16 2.220446049250313e-16 1.0 0.20000000000000007; -1.0 0.0 2.220446049250313e-16 0.10000000000000005; 0.0 0.0 0.0 1.0]
i:5Tlist[4.930380657631324e-32 -1.0 2.220446049250313e-16 1.1102230246251565e-16; 2.220446049250313e-16 2.220446049250313e-16 1.0 0.5; -1.0 0.0 2.220446049250313e-16 0.10000000000000012; 0.0 0.0 0.0 1.0]
i:6Tlist[4.930380657631324e-32 -1.0 2.220446049250313e-16 1.3322676295501878e-16; 2.220446049250313e-16

3-element Vector{Float64}:
 1.3322676295501878e-16
 0.6
 0.10000000000000014

In [ ]:



function pendulum_dynamics(x)
    l = 1.0
    g = 9.81

    θ = x[1]
    θ̇ = x[2]

    θ̈ = -(g / l) * sin(θ)

    return [θ̇; θ̈]
end

In [ ]:
function backward_euler_step_fixed_point(fun, x0, h)
    xn = x0
    e = [norm(x0 + h .* fun(xn) - xn)]
    while e[end] > 1e-8
        xn = x0 + h .* fun(xn)
        e = [e; norm(x0 + h .* fun(xn) - xn)]
    end

    return xn, e
end

In [ ]:
function backward_euler_step_newton(fun, x0, h)
    xn = x0
    r = x0 + h .* fun(xn) - xn
    e = [norm(r)]
    while e[end] > 1e-8
        ∂r = ForwardDiff.jacobian(x -> x0 + h .* fun(x) - x, xn)
        xn = xn - ∂r \ r
        r = x0 + h .* fun(xn) - xn
        e = [e; norm(r)]
    end

    return xn, e
end

In [ ]:
function backward_euler_fixed_point(fun, x0, Tf, h)
    t = Array(range(0, Tf, step=h))

    x_hist = zeros(length(x0), length(t))
    x_hist[:, 1] .= x0

    for k = 1:(length(t)-1)
        x_hist[:, k+1], e = backward_euler_step_fixed_point(fun, x_hist[:, k], h)
    end

    return x_hist, t
end

In [ ]:
function backward_euler_newton(fun, x0, Tf, h)
    t = Array(range(0, Tf, step=h))

    x_hist = zeros(length(x0), length(t))
    x_hist[:, 1] .= x0

    for k = 1:(length(t)-1)
        x_hist[:, k+1], e = backward_euler_step_newton(fun, x_hist[:, k], h)
    end

    return x_hist, t
end

In [ ]:
x0 = [0.1; 0]
x_hist1, t_hist1 = backward_euler_fixed_point(pendulum_dynamics, x0, 10, 0.01)
x_hist2, t_hist2 = backward_euler_newton(pendulum_dynamics, x0, 10, 0.01)
plot(t_hist1, x_hist1[1, :])
plot(t_hist2, x_hist2[1, :])

In [ ]:
max(abs.(x_hist1 - x_hist2)...)

In [ ]:
xn, e1 = backward_euler_step_fixed_point(pendulum_dynamics, x0, 0.1)
e1

In [ ]:
xn, e2 = backward_euler_step_newton(pendulum_dynamics, x0, 0.1)
e2

In [ ]:
semilogy(e1)
semilogy(e2)